# Gold Company Activity Mart

This notebook creates the company-level hiring activity data mart aggregated by sector.

## Architecture

**Input**: `workspace.warehouse.fact_job_postings`  
**Output**: `workspace.gold.gold_company_activity`  
**Mode**: Full refresh (CREATE OR REPLACE)

## Schema

**Grain**: Sector × Company (snapshot of current activity)  
**Primary Key**: (sector_sk, company_sk)  
**Partitioning**: sector_sk

**Key Metrics**:
- `active_jobs`: Current active jobs
- `total_jobs_30d`: Jobs last 30 days  
- `top_role`: Most hired role
- `updated_at`: Last refresh timestamp

In [0]:
# Databricks notebook parameters
dbutils.widgets.text("lookback_days", "365", "Lookback Days")
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")

# Get parameter values
lookback_days = int(dbutils.widgets.get("lookback_days"))
force_full_refresh = dbutils.widgets.get("force_full_refresh").lower() == "true"

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

# Configuration
CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
GOLD_SCHEMA = f"{CATALOG}.gold"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Current run metadata
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()
cutoff_date = int((datetime.now() - timedelta(days=lookback_days)).strftime("%Y%m%d"))

print(f"Run ID: {run_id}")
print(f"Lookback days: {lookback_days}")
print(f"Cutoff date: {cutoff_date}")
print(f"Force full refresh: {force_full_refresh}")

In [0]:
# Create metadata table to track refresh runs
metadata_table = f"{METADATA_SCHEMA}.gold_company_activity_refresh_log"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  run_id STRING,
  run_timestamp TIMESTAMP,
  lookback_days INT,
  cutoff_date INT,
  rows_created BIGINT,
  unique_sectors BIGINT,
  unique_companies BIGINT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DOUBLE,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks refresh runs for gold_company_activity'
""")

print(f"Metadata table ready: {metadata_table}")

In [0]:
import time

start_time = time.time()

print("Creating gold_company_activity mart...")

# Execute CREATE OR REPLACE
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_SCHEMA}.gold_company_activity
USING DELTA
PARTITIONED BY (sector_sk)
COMMENT 'Company hiring activity metrics by sector'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
AS

WITH recent_30d AS (
  SELECT
    f.sector_sk,
    f.company_sk,
    f.role_sk,
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(*) AS total_jobs
  FROM {WAREHOUSE_SCHEMA}.fact_job_postings f
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 30), 'yyyyMMdd') AS INT)
  GROUP BY f.sector_sk, f.company_sk, f.role_sk
),

top_roles AS (
  SELECT
    sector_sk,
    company_sk,
    role_sk,
    active_jobs,
    ROW_NUMBER() OVER (PARTITION BY sector_sk, company_sk ORDER BY active_jobs DESC, role_sk) AS rn
  FROM recent_30d
),

aggregated AS (
  SELECT
    r.sector_sk,
    r.company_sk,
    SUM(r.active_jobs) AS active_jobs,
    SUM(r.total_jobs) AS total_jobs_30d,
    FIRST(dr.role_name) AS top_role
  FROM recent_30d r
  LEFT JOIN top_roles tr ON r.sector_sk = tr.sector_sk AND r.company_sk = tr.company_sk AND tr.rn = 1
  LEFT JOIN {WAREHOUSE_SCHEMA}.dim_role dr ON tr.role_sk = dr.role_sk
  GROUP BY r.sector_sk, r.company_sk
)

SELECT
  sector_sk,
  company_sk,
  active_jobs,
  total_jobs_30d,
  top_role,
  CURRENT_TIMESTAMP() AS updated_at
FROM aggregated
ORDER BY sector_sk, active_jobs DESC
""")

processing_time = time.time() - start_time
print(f"✓ Table created successfully in {processing_time:.2f} seconds")

In [0]:
# Get table statistics
table_stats = spark.sql(f"""
SELECT
  COUNT(*) AS rows_created,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  COUNT(DISTINCT company_sk) AS unique_companies
FROM {GOLD_SCHEMA}.gold_company_activity
""").collect()[0]

rows_created = table_stats["rows_created"]
unique_sectors = table_stats["unique_sectors"]
unique_companies = table_stats["unique_companies"]

# Log metadata - match table schema from cell 4
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType, TimestampType, BooleanType

metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("run_timestamp", TimestampType(), True),
    StructField("lookback_days", IntegerType(), True),
    StructField("cutoff_date", IntegerType(), True),
    StructField("rows_created", LongType(), True),
    StructField("unique_sectors", LongType(), True),
    StructField("unique_companies", LongType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processing_time_seconds", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

metadata_df = spark.createDataFrame([(
    run_id,
    run_timestamp,
    lookback_days,
    cutoff_date,
    rows_created,
    unique_sectors,
    unique_companies,
    force_full_refresh,
    processing_time,
    "success",
    None
)], schema=metadata_schema)

metadata_df.write.format("delta").mode("append").saveAsTable(metadata_table)

print(f"\nRefresh Summary:")
print(f"  Run ID: {run_id}")
print(f"  Rows created: {rows_created:,}")
print(f"  Unique sectors: {unique_sectors}")
print(f"  Unique companies: {unique_companies}")
print(f"  Processing time: {processing_time:.2f}s")

In [0]:
%sql
-- Validation: Summary Statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  COUNT(DISTINCT company_sk) AS unique_companies,
  SUM(active_jobs) AS total_active_jobs,
  SUM(total_jobs_30d) AS total_jobs_30d,
  ROUND(AVG(active_jobs), 2) AS avg_active_jobs,
  MAX(updated_at) AS last_refreshed
FROM workspace.gold.gold_company_activity;